# SETUP

In [2]:
#config
import json
from pathlib import Path
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2
import sys

project_root = Path.cwd()
while not (project_root / "src" / "A_Config.py").exists():
    if project_root == project_root.parent:
        raise RuntimeError("couldn't find repo root (src/A_Config.py) above cwd")
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
from A_Config import (
    set_case, case_dir, source_dir, query_dir, sam3_masks_dir,
    frames_for_cam_poses_dir, frames_for_recon_dir, vggt_o_output_dir,
    predictions_path, ply_dir, reg_dir, csv_path_gps, csv_path_pose,
    assets_dir, recon_for_MAP_dir,recon_for_EVENT_dir, agent_p_output_dir,
    set_pass, pass_num
)
from Eval_defs import random_baseline_closed_form,random_baseline_blocks, load_labels,paper_edit_to_timegrid,score_paper_vs_GT_edits,labels_dir,cut_csv_to_timegrid, time_grid,spans_to_mask, annotator_beats_to_grid   # now resolvable
 


case_name =  "304_met_multi_4" 
experiment = "Gem_merge_test"
set_case(case_name, experiment)
paper_edit_json_path = agent_p_output_dir() / f"304_met_multi_4_paper_edit_draft-1.json"
my_cut_csv = labels_dir() / "304_met_multi_4_cuts.csv"
BIN_S = 1.0      # vote grid, seconds. 0.1 matches the label precision if you want it finer
WEIGHTS = None   # or {"important": 1.0, "critical": 2.0}


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# MY CUT
## VS machine

In [3]:
edges = time_grid(case_name, bin_s=BIN_S, labels=None, tol_s=1.0)
machine_paper_edit = paper_edit_to_timegrid(paper_edit_json_path, case_name, edges)
GT_edit = cut_csv_to_timegrid(my_cut_csv, edges)   # w, 0 or 1
#beta controls f_beta wihc controls soft f1
score_paper_vs_GT_edits(GT_edit, machine_paper_edit)
#f beta is precision and recall together. the F measure
#f-1 is beta =1 same calc but hard coded



304_met_multi_4: 2/3 beats have no resolved frames — skipped


/workspace/project/src/Experiments/system_eval/Eval_defs.py:140: RuntimeWarning: Mean of empty slice.
  soft_specificity =float(np.where(paper_edit, soft_gt,1.0)[~GT_edit].mean())
/home/vscode/.local/lib/python3.11/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


{'soft_precision': 1.0,
 'precision': 1.0,
 'soft_recall': 1.0,
 'recall': 1.0,
 'soft_specificity': nan,
 'specificity': nan,
 'iou': 1.0,
 'soft_f1': 1.0,
 'f_beta': 1.0,
 'beta': 1.0,
 'decay_s': 4.0,
 'true_p_s': 25.0,
 'false_p_s': 0.0,
 'false_n_s': 0.0,
 'true_n_s': 0.0,
 'edit_s': 25.0,
 'gt_s': 25.0,
 'len_ratio': 1.0,
 'n_segments': 1,
 'shortest_seg_s': 25.0,
 'median_seg_s': 25.0}

# VS humans

In [5]:
decay_s=4
rows = [dict(editor="ME (GT)", time_min=70.0,
             **score_paper_vs_GT_edits(GT_edit, GT_edit, decay_s=decay_s))]

rows.append(dict(editor="machine", time_min=np.nan,
                 **score_paper_vs_GT_edits(GT_edit, machine_paper_edit, decay_s=decay_s)))

#for l in load_labels():
#    rows.append(dict(editor=l["annotator"],
#                     time_min=round(l["time_spent_sec"] / 60, 1),
#                     **score_paper_vs_GT_edits(GT_edit, annotator_beats_to_grid(l, edges), decay_s=decay_s)))
rand_row, null_stats = random_baseline_blocks(GT_edit, machine_paper_edit)
rows.append(rand_row)
rows.append(dict(editor="whole clip", time_min=np.nan,
                 **score_paper_vs_GT_edits(GT_edit, np.ones_like(GT_edit), decay_s=decay_s)))
comp = pd.DataFrame(rows)
comp[["editor", "time_min", "soft_f1",
      "soft_specificity", "specificity",
      "soft_precision", "precision",
      "soft_recall", "recall",
      "iou", "len_ratio", "n_segments", "median_seg_s", "edit_s"]].round(3)



/workspace/project/src/Experiments/system_eval/Eval_defs.py:216: RuntimeWarning: Mean of empty slice
  row = {k: float(np.nanmean([dr[k] for dr in draws])) for k in num_keys}
/workspace/project/src/Experiments/system_eval/Eval_defs.py:224: RuntimeWarning: Mean of empty slice
  mean   = float(np.nanmean(vals)),
/home/vscode/.local/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1406: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/workspace/project/src/Experiments/system_eval/Eval_defs.py:140: RuntimeWarning: Mean of empty slice.
  soft_specificity =float(np.where(paper_edit, soft_gt,1.0)[~GT_edit].mean())
/home/vscode/.local/lib/python3.11/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,editor,time_min,soft_f1,soft_specificity,specificity,soft_precision,precision,soft_recall,recall,iou,len_ratio,n_segments,median_seg_s,edit_s
0,ME (GT),70.0,1.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,25.0,25.0
1,machine,NaN,1.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,25.0,25.0
2,"random (blocks, n=200)",NaN,1.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,25.0,25.0
3,whole clip,NaN,1.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,25.0,25.0


In [6]:
null_stats

,Producer_agent_score,mean,p05,p95,pct
iou,1.0,1.0,1.0,1.0,0.0
soft_f1,1.0,1.0,1.0,1.0,0.0
precision,1.0,1.0,1.0,1.0,0.0
recall,1.0,1.0,1.0,1.0,0.0
specificity,NaN,NaN,NaN,NaN,0.0


In [ ]:
cols = ["editor", "time_min", "edit_s", "len_ratio",
        "soft_f1", "soft_precision", "soft_recall", "soft_specificity",
        "iou", "n_segments", "median_seg_s"]

nice = {"editor": "Editor", "time_min": "Time (min)", "edit_s": "Cut (s)",
        "len_ratio": "Len ratio", "soft_f1": "Soft $F_1$",
        "soft_precision": "Soft prec.", "soft_recall": "Soft rec.",
        "soft_specificity": "Soft spec.", "iou": "IoU",
        "n_segments": "Shots", "median_seg_s": "Median shot (s)"}

(comp[cols].rename(columns=nice)
 .to_latex(project_root / "writing" /"Figures" / "sys_eval"/ "tab_noodles_comp.tex",
           index=False, escape=False, float_format="%.3f",
           column_format="lrrrrrrrrrr",
           caption="Editors scored against the reference cut, 503\\_EGO\\_noodles.",
           label="tab:noodles_comp",
           position="htbp"))
